In [ ]:
# Install Dependencies
!pip install -q langchain faiss-cpu pypdf langchain-community
!pip install -q sentence-transformers
!pip install -q ollama
!pip install -q langchain-ollama
!pip install -q langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# Import Libraries
# Document loaders
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector database
from langchain_community.vectorstores import FAISS

# Ollama LLM + Embedding
from langchain_ollama import ChatOllama, OllamaEmbeddings

# New RAG chain, instead of RetrievalQA
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
import bs4
# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://docs.python.org/3/whatsnew/3.14.html",),

)
docs = loader.load()


In [ ]:

print(f"Loaded {len(docs)} documents")


Loaded 1 documents


In [ ]:
print(docs)

[Document(metadata={'source': 'https://docs.python.org/3/whatsnew/3.14.html', 'title': 'What‚Äôs new in Python 3.14 — Python 3.14.6 documentation', 'description': 'Editors, Adam Turner and Hugo van Kemenade,. This article explains the new features in Python 3.14, compared to 3.13. Python 3.14 was released on 7 October 2025. For full details, see the changelog...', 'language': 'en'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nWhat‚Äôs new in Python 3.14 — Python 3.14.6 documentation\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n    Theme\n    \nAuto\nLight\nDark\n\n\n\nTable of Contents\n\nWhat‚Äôs new in Python 3.14\nSummary ‚Äì Release highlights\nNew features\nPEP 649 & PEP 749: Deferred evaluation of annotations\nPEP 734: Multiple interpreters in the standard library\nPEP 750: Template string literals\nPEP 768: Safe external debugger interface\nA new type of interpreter\nFree-threaded mode improvements\nImproved error m

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")


Split into 178 chunks


In [ ]:
for index, i in enumerate(chunks[:2]):
  print(f"Chunk Number ",index+1)
  print(i)
  print("#####################")

Chunk Number  1
page_content='What‚Äôs new in Python 3.14 — Python 3.14.6 documentation




















































    Theme
    
Auto
Light
Dark



Table of Contents

What‚Äôs new in Python 3.14
Summary ‚Äì Release highlights
New features
PEP 649 & PEP 749: Deferred evaluation of annotations
PEP 734: Multiple interpreters in the standard library
PEP 750: Template string literals
PEP 768: Safe external debugger interface
A new type of interpreter
Free-threaded mode improvements
Improved error messages
PEP 784: Zstandard support in the standard library
Asyncio introspection capabilities
Concurrent safe warnings control


Other language changes
Built-ins
Command line and environment
PEP 758: Allow except and except* expressions without brackets
PEP 765: Control flow in finally blocks
Garbage collection
Default interactive shell' metadata={'source': 'https://docs.python.org/3/whatsnew/3.14.html', 'title': 'What‚Äôs new in Python 3.14 — Python 3.14.6 documentation'

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,308 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Pack

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118265 files and directories currently installed.)
Preparing to unpack .../libcurl4-openssl-dev_7.81.0-1ubuntu1.25_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.25) over (7.81.0-1ubuntu1.24) ...
Preparing to unpack .../curl_7.81.0-1ubuntu1.25_amd64.deb ...
Unpacking curl (7.81.0-1ubuntu1.25) over (7.81.0-1ubuntu1.24) ...
Preparing to unpack .../libcurl4_7.81.0-1ubuntu1.25_amd64.deb ...
Unpacking libcurl4:amd64 (7.81.0-1ubuntu1.25) over (7.81.0-1ubuntu1.24) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up libcurl4:amd64 (7.81.0-1ubuntu1.25) ...
Setting up curl (7.81.0-1ubuntu1.25) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Setting up libcurl4-o

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull all-minilm

In [ ]:
!ollama pull gemma3

In [ ]:
!pip install -U langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [ ]:
llm=ChatOllama(
    model="gemma3",
    temperature=0.5
)

In [ ]:
embedding_model=OllamaEmbeddings(model="all-minilm")

In [ ]:
!pip install -q -U \
  langchain \
  langchain-core \
  langchain-classic \
  langchain-community \
  langchain-text-splitters \
  langchain-ollama \
  faiss-cpu \
  pypdf \
  bs4 \
  requests==2.32.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 919.1 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import requests
print("requests:", requests.__version__)

from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama, OllamaEmbeddings

print("✅ Imports working")

requests: 2.34.2
✅ Imports working


In [ ]:
import os
os.environ["USER_AGENT"] = "my-rag-app/1.0"

from langchain_community.document_loaders import WebBaseLoader

url = "https://example.com"

loader = WebBaseLoader(url)
documents = loader.load()

print("✅ Documents loaded:", len(documents))

✅ Documents loaded: 1


In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
print("✅ Vectorstore created with FAISS")

✅ Vectorstore created with FAISS


In [ ]:
vectorstore.similarity_search("python documentation",k=5)

[Document(id='9d02b30a-332a-4c51-ae2f-50ab56522bc2', metadata={'source': 'https://docs.python.org/3/whatsnew/3.14.html', 'title': 'What‚Äôs new in Python 3.14 — Python 3.14.6 documentation', 'description': 'Editors, Adam Turner and Hugo van Kemenade,. This article explains the new features in Python 3.14, compared to 3.13. Python 3.14 was released on 7 October 2025. For full details, see the changelog...', 'language': 'en'}, page_content='Summary ‚Äì Release highlights¬∂\nPython 3.14 is the latest stable release of the Python programming\nlanguage, with a mix of changes to the language, the implementation,\nand the standard library.\nThe biggest changes include template string literals,\ndeferred evaluation of annotations,\nand support for subinterpreters in\nthe standard library.\nThe library changes include significantly improved capabilities for\nintrospection in asyncio,\nsupport for Zstandard via a new\ncompression.zstd module, syntax highlighting in the REPL,\nas well as the usua

In [ ]:
vectorstore.similarity_search("version.txt",k=5)

[Document(id='3fcfe4d9-d84f-4064-aced-87979cb09529', metadata={'source': 'https://docs.python.org/3/whatsnew/3.14.html', 'title': 'What‚Äôs new in Python 3.14 — Python 3.14.6 documentation', 'description': 'Editors, Adam Turner and Hugo van Kemenade,. This article explains the new features in Python 3.14, compared to 3.13. Python 3.14 was released on 7 October 2025. For full details, see the changelog...', 'language': 'en'}, page_content="build-details.json¬∂\nInstallations of Python now contain a new file, build-details.json.\nThis is a static JSON document containing build details for CPython,\nto allow for introspection without needing to run code.\nThis is helpful for use-cases such as Python launchers, cross-compilation,\nand so on.\nbuild-details.json must be installed in the platform-independent\nstandard library directory. This corresponds to the ‚Äòstdlib‚Äô sysconfig installation path,\nwhich can be found by running sysconfig.get_path('stdlib').\n\nSee also\nPEP 739 ‚Äì build

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

system_prompt = """
You are a helpful assistant.
Use only the given context to answer the question.
If the answer is not available in the context, say:
"I don't know based on the provided document."

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

document_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, document_chain)

print("✅ QA chain created successfully")

✅ QA chain created successfully


In [ ]:
question = "What is the latest version of Python and why was it created?"

result = qa_chain.invoke({"input": question})

print("❓ Question:", question)
print("📎 Answer:", result["answer"])

❓ Question: What is the latest version of Python and why was it created?
📎 Answer: According to the document, the latest stable release of the Python programming language is Python 3.14. It was released on 7 October 2025. The document doesn't state *why* it was created, only that it has a mix of changes to the language, implementation, and standard library.


In [ ]:
questions = [
    "Explain multiple interpreters and their role in concurrency.",
"Compare multiple interpreters with multiprocessing.",

"Write the limitations of multiple interpreters in Python 3.14.",
"List the major interpreter improvements in Python 3.14.",
"Explain how Python 3.14 improves standard library support."
    # "What is an LLM-powered autonomous agent?",
    # "How does the planning component work in such agents?",
    # "What is an Agent",
    # "What is the Chain of Thought (CoT) technique and why is it useful?",
    # "How does Tree of Thoughts improve upon Chain of Thought?",
    # "What is LLM+P and how does it involve external planners?"
]

In [ ]:
for question in questions:
    print(f"\n❓ Question: {question}")

    result = qa_chain.invoke({"input": question})

    print("📎 Answer:", result["answer"])

    print("\n📚 Sources:")
    for i, doc in enumerate(result["context"], start=1):
        print(f"\nSource {i}")
        print("Metadata:", doc.metadata)
        print("Preview:", doc.page_content[:300])


❓ Question: Explain multiple interpreters and their role in concurrency.
📎 Answer: Multiple interpreters offer a way to achieve concurrency by creating isolated logical “processes” that can run in parallel, with no sharing by default. They’re similar to threads in terms of efficiency but provide the isolation of processes. As of Python 3.12, they are now sufficiently isolated to be used for CPU-intensive tasks, unlocking use cases previously limited by the GIL.

📚 Sources:

Source 1
Metadata: {'source': 'https://docs.python.org/3/whatsnew/3.14.html', 'title': 'What‚Äôs new in Python 3.14 — Python 3.14.6 documentation', 'description': 'Editors, Adam Turner and Hugo van Kemenade,. This article explains the new features in Python 3.14, compared to 3.13. Python 3.14 was released on 7 October 2025. For full details, see the changelog...', 'language': 'en'}
Preview: For some use cases, concurrency in software improves efficiency and
can simplify design, at a high level.
At the same time, im